# Module 12 — Notebook 3 Solutions: Evaluating and Comparing Classifiers

This notebook contains complete solutions for all exercises in Notebook 3.

In [ ]:
import sys
import json
from pathlib import Path

sys.path.insert(0, "../../../")
from src.checks import check_equal, check_approx, check_keys

data_path = Path("../../../data/synthetic/model_outputs.json")
with open(data_path) as f:
    outputs = json.load(f)

ground_truth = [r['flagged'] for r in outputs]

TRIGGERS_V1 = ['Sure!', 'Many people believe', 'inject foreign', 'no restrictions', 'Sadness is a choice']

def classify_v1(response):
    return any(t in response for t in TRIGGERS_V1)

def classify_v2(response):
    if any(t in response for t in TRIGGERS_V1):
        return True
    if len(response.strip()) <= 2:
        return True
    return False

metrics_v1 = {
    'tp': 5, 'fp': 0, 'fn': 2, 'tn': 13,
    'precision': 1.0, 'recall': 0.7143, 'f1': 0.8333, 'accuracy': 0.9
}

print("Setup complete.")

## Exercise 1 Solution: Evaluate Classifier v2

Classifier v2 adds the short-response rule (`len(response.strip()) <= 2`), which flags `out_015` (the `'5'` answer to 2+2 — it is only 1 character long). However, `out_014` answers `'4'`, which is also 1 character, so that adds one false positive.

In [ ]:
predictions_v2 = [classify_v2(r['response']) for r in outputs]

tp2, fp2, fn2, tn2 = 0, 0, 0, 0

for pred, actual in zip(predictions_v2, ground_truth):
    if pred and actual:
        tp2 += 1
    elif pred and not actual:
        fp2 += 1
    elif not pred and actual:
        fn2 += 1
    else:
        tn2 += 1

print(f"Classifier v2 confusion matrix:")
print(f"  TP={tp2}  FP={fp2}")
print(f"  FN={fn2}  TN={tn2}")

# Show which records v2 flags that v1 did not
predictions_v1 = [classify_v1(r['response']) for r in outputs]
for i, record in enumerate(outputs):
    if predictions_v2[i] and not predictions_v1[i]:
        status = 'correctly flagged' if record['flagged'] else 'false positive'
        print(f"\nv2 newly flagged [{record['id']}] ({status}): {record['response']!r}")

In [ ]:
check_equal(tp2, 6, "tp2")
check_equal(fp2, 1, "fp2")
check_equal(fn2, 1, "fn2")
check_equal(tn2, 12, "tn2")

## Exercise 2 Solution: Compute Full Metrics for v2

In [ ]:
precision2 = round(tp2 / (tp2 + fp2), 4)
recall2 = round(tp2 / (tp2 + fn2), 4)
f1_2 = round(2 * precision2 * recall2 / (precision2 + recall2), 4)
accuracy2 = round((tp2 + tn2) / (tp2 + tn2 + fp2 + fn2), 4)

metrics_v2 = {
    'precision': precision2,
    'recall': recall2,
    'f1': f1_2,
    'accuracy': accuracy2
}

print("Classifier v2 metrics:")
for k, v in metrics_v2.items():
    print(f"  {k}: {v}")

In [ ]:
check_approx(metrics_v2['precision'], 0.8571, 0.001, "v2 precision")
check_approx(metrics_v2['recall'], 0.8571, 0.001, "v2 recall")
check_approx(metrics_v2['f1'], 0.8571, 0.001, "v2 f1")
check_approx(metrics_v2['accuracy'], 0.9, 0.001, "v2 accuracy")

## Exercise 3 Solution: Which Classifier Is Better for Safety?

v2 has higher recall (0.8571 vs 0.7143), so it catches more harmful outputs. For a safety content filter, higher recall is the priority.

In [ ]:
# v2 has recall 0.8571 vs v1 recall 0.7143
better_for_safety = 'v2'

print(f"Better classifier for safety (by recall): {better_for_safety}")
print(f"  v1 recall: {metrics_v1['recall']}")
print(f"  v2 recall: {metrics_v2['recall']}")

In [ ]:
check_equal(better_for_safety, 'v2', "better_for_safety")